### Homework: Distributed Training and Scaling Inference

#### Instructions:
- Complete the following exercises based on the lecture material.
- Submit your completed notebook with all cells executed.

---

## Part 1: Distributed Training Strategies

### Question 1: Understanding Parallelism
**(a)** Compare data parallelism and model parallelism. When would you use each approach?

- Data parallelism improves training speed for models that can fit in a single GPU's memory by using many copies of the model and training each one on parts of the dataset. The models synchronize gradients between batches
- Model parallelism splits a single instance of the model across many devices to handle very large parameter counts which cannot fit in a single device's memory.
- Data parallelism is appropriate for iterating on smaller models rapidly, while model parallelism is used for training very large models. Aspects of the two approaches can be combined to train the largest models effeciently.

**(b)** Explain pipeline parallelism and describe its advantages over traditional model parallelism.

- Pipeline parallelism runs one layer of a large model on each GPU, decreasing the memory requirements per device.
- This method is relatively simple to implement and has less communication overhead.


### Question 2: Implementing Data Parallel Training
Modify the given function to implement data parallel training using PyTorch’s DistributedDataParallel (DDP).

In [8]:
%%writefile ddp_demo.py
import os
import argparse
import torch
import torch.nn as nn
import torch.optim as optim
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP

def setup(backend: str):
    dist.init_process_group(backend=backend, init_method="env://")
    rank = dist.get_rank()
    world_size = dist.get_world_size()
    local_rank = int(os.environ.get("LOCAL_RANK", 0))
    return rank, world_size, local_rank

def main():
    p = argparse.ArgumentParser()
    p.add_argument("--backend", default="gloo", choices=["gloo", "nccl"])
    p.add_argument("--device", default="cpu", choices=["cpu", "cuda"])
    p.add_argument("--steps", type=int, default=20)
    args = p.parse_args()

    rank, world_size, local_rank = setup(args.backend)

    if args.device == "cuda":
        torch.cuda.set_device(local_rank)
        device = torch.device("cuda", local_rank)
    else:
        device = torch.device("cpu")

    # different seed per rank
    torch.manual_seed(1234 + rank)

    model = nn.Linear(10, 1).to(device)

    if args.device == "cuda":
        model = DDP(model, device_ids=[local_rank], output_device=local_rank)
    else:
        model = DDP(model)

    opt = optim.SGD(model.parameters(), lr=0.01)
    loss_fn = nn.MSELoss()

    for step in range(args.steps):
        x = torch.randn(32, 10, device=device)
        y = torch.randn(32, 1, device=device)
        opt.zero_grad(set_to_none=True)
        loss = loss_fn(model(x), y)
        loss.backward()
        opt.step()

        print(f"[rank {rank+1}/{world_size}] step={step} loss={loss.item():.4f}")

    dist.destroy_process_group()

if __name__ == "__main__":
    main()

Overwriting ddp_demo.py


In [9]:
!torchrun --standalone --nproc_per_node=2 ddp_demo.py --backend gloo --device cpu --steps 30

W0301 20:27:08.451000 2146 torch/distributed/run.py:852] 
W0301 20:27:08.451000 2146 torch/distributed/run.py:852] *****************************************
W0301 20:27:08.451000 2146 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0301 20:27:08.451000 2146 torch/distributed/run.py:852] *****************************************
[Gloo] Rank [Gloo] Rank 0 is connected to 1 peer ranks. Expected number of connected peer ranks is : 1
1 is connected to 1 peer ranks. Expected number of connected peer ranks is : 1
[rank 2/2] step=0 loss=0.9271[rank 1/2] step=0 loss=1.4412

[rank 2/2] step=1 loss=1.4785
[rank 1/2] step=1 loss=1.0326
[rank 2/2] step=2 loss=1.2990[rank 1/2] step=2 loss=1.0545

[rank 1/2] step=3 loss=1.1503
[rank 2/2] step=3 loss=0.8874
[rank 1/2] step=4 loss=0.8795
[rank 2/2] step=4 loss=1.

Modify the script to support multi-node training with `torchrun`.

---

## Part 2: Challenges and Libraries for Distributed Training

### Question 3: Overcoming Challenges in Distributed Training
**(a)** What are the major challenges in distributed training, and how can they be mitigated?

- Communication overhead, can be overcome by overlapping compute with communication
- Memory limitations can be addressed by mixed precision
- Fault tolerance is solved by elastic training


**(b)** Compare PyTorch DDP, DeepSpeed, and Fully Sharded Data Parallel (FSDP). When should each be used?

- PyTorch DDP improves training speed but does not address memory limitations. It is simple and works well for models that fit in a single device's memory.
- FSDP splits parameters, gradients, and activations across multiple devices. Works well for models that do not fit in a single device's memory, but still integrtes well with PyTorch.
- DeepSpeed is highly optimized for very large models, but is more of a seperate framework compared to PyTorch.

### Question 4: Implementing FSDP for Large Model Training
Modify the given script to train a large model using PyTorch FSDP.

In [2]:
%%writefile train_fsdp_amp.py
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torch.distributed as dist

from torch.amp import autocast, GradScaler
from torch.distributed.fsdp import FullyShardedDataParallel as FSDP
from torch.distributed.fsdp.wrap import wrap


def train_fsdp_model():
    # torchrun sets these
    rank = int(os.environ["RANK"])
    world_size = int(os.environ["WORLD_SIZE"])
    local_rank = int(os.environ.get("LOCAL_RANK", 0))

    use_cuda = torch.cuda.is_available()
    backend = "nccl" if use_cuda else "gloo"

    dist.init_process_group(backend=backend)

    if use_cuda:
        torch.cuda.set_device(local_rank)
        device = torch.device(f"cuda:{local_rank}")
        amp_device = "cuda"
        amp_dtype = torch.float16
        scaler = GradScaler(device="cuda")
    else:
        device = torch.device("cpu")
        amp_device = "cpu"
        amp_dtype = torch.bfloat16 # CPU works better with bf16
        scaler = None # scaling is not needed for CPU

    model = wrap(nn.Linear(1000, 1000).to(device))
    model = FSDP(model)

    optimizer = optim.AdamW(model.parameters(), lr=1e-4)
    loss_fn = nn.MSELoss()

    for _ in range(50):
        x = torch.randn(32, 1000, device=device)
        y = torch.randn(32, 1000, device=device)
        optimizer.zero_grad(set_to_none=True)

        with autocast(device_type=amp_device, dtype=amp_dtype):
            output = model(x)
            loss = loss_fn(output, y)

        if scaler is None:
            loss.backward()
            optimizer.step()
        else:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

        if rank == 0:
            print("loss:", float(loss))

    dist.destroy_process_group()


if __name__ == "__main__":
    train_fsdp_model()

Writing train_fsdp_amp.py


In [4]:
!torchrun --standalone --nproc_per_node=1 train_fsdp_amp.py

/usr/local/lib/python3.12/dist-packages/torch/distributed/fsdp/fully_sharded_data_parallel.py:479: UserWarning: FSDP is switching to use `NO_SHARD` instead of ShardingStrategy.FULL_SHARD since the world size is 1.
  _init_core_state(
/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: The AccumulateGrad node's stream does not match the stream of the node that produced the incoming gradient. This may incur unnecessary synchronization and break CUDA graph capture if the AccumulateGrad node's stream is the default stream. This mismatch is caused by an AccumulateGrad node created prior to the current iteration being kept alive. This can happen if the autograd graph is still being kept alive by tensors such as the loss, or if you are using DDP, which will stash a reference to the node. To resolve the mismatch, delete all references to the autograd graph or ensure that DDP initialization is performed under the same stream as subsequent forwards. If the mismatch 

Modify the script to enable mixed precision training using `torch.amp`.

---

## Part 3: Scaling Inference and Deployment

### Question 5: Optimizing Inference Performance
**(a)** What are some key optimizations used in model inference for reducing latency and memory consumption?

- Inference optimizations include reduced precision, fused operations, and GPU kernels designed for the best transformer performance.

**(b)** Explain the trade-offs between batch inference and real-time streaming inference.

- Batch inference combines multiple requests into a single GPU operation which improves throughput and effeciency.
- The tradeoff when using batch inference is that some requests will need to wait until a full batch is ready to compute, which increases the latency, and has a negative impact on real-time streaming.

### Question 6: Deploying a Model with TorchServe
Modify the given script to package a trained model for deployment using TorchServe.

In [7]:
%%writefile model.py
import torch.nn as nn

class SimpleModel(nn.Module):
    def __init__(self, in_features: int = 10):
        super().__init__()
        self.fc = nn.Linear(in_features, 1)

    def forward(self, x):
        return self.fc(x)

Writing model.py


In [8]:
%%writefile save_model.py
import torch
from model import SimpleModel

def main():
    torch.manual_seed(0)
    model = SimpleModel()
    model.eval()
    torch.save(model.state_dict(), "model.pth")
    print("Saved model weights to model.pth")

if __name__ == "__main__":
    main()

Overwriting save_model.py


In [12]:
%%writefile handler.py
import json
import logging
import os
from typing import Any, List

import torch
from ts.torch_handler.base_handler import BaseHandler
from model import SimpleModel

logger = logging.getLogger(__name__)

class SimpleModelHandler(BaseHandler):
    def __init__(self):
        super().__init__()
        self.initialized = False
        self.device = torch.device("cpu")
        self.model = None

    def initialize(self, context):
        properties = context.system_properties
        model_dir = properties.get("model_dir")
        gpu_id = properties.get("gpu_id")

        if torch.cuda.is_available() and gpu_id is not None:
            self.device = torch.device(f"cuda:{gpu_id}")
        else:
            self.device = torch.device("cpu")

        manifest = context.manifest
        serialized_file = manifest["model"]["serializedFile"]
        weights_path = os.path.join(model_dir, serialized_file)

        self.model = SimpleModel(in_features=10)
        state = torch.load(weights_path, map_location=self.device)
        self.model.load_state_dict(state)
        self.model.to(self.device)
        self.model.eval()
        self.initialized = True

    def _decode(self, payload: Any) -> Any:
        if isinstance(payload, (bytes, bytearray)):
            payload = payload.decode("utf-8")
        if isinstance(payload, str):
            return json.loads(payload)
        return payload

    def preprocess(self, data: List[dict]) -> torch.Tensor:
        row0 = data[0]
        payload = row0.get("data") or row0.get("body")
        payload = self._decode(payload)

        if isinstance(payload, dict):
            payload = payload.get("data") or payload.get("instances") or payload.get("inputs")

        if isinstance(payload, list) and len(payload) == 10 and all(isinstance(x, (int, float)) for x in payload):
            batch = [payload]
        elif isinstance(payload, list) and payload and isinstance(payload[0], list):
            batch = payload
        else:
            raise ValueError("Send {'data':[10 floats]} or {'instances':[[10 floats], ...]}")

        for vec in batch:
            if len(vec) != 10:
                raise ValueError("Expected 10 features")

        return torch.tensor(batch, dtype=torch.float32, device=self.device)

    @torch.inference_mode()
    def inference(self, x: torch.Tensor) -> torch.Tensor:
        return self.model(x)

    def postprocess(self, y: torch.Tensor):
        return y.detach().cpu().view(-1).tolist()

    def handle(self, data, context):
        x = self.preprocess(data)
        y = self.inference(x)
        return self.postprocess(y)

Writing handler.py


In [16]:
%%writefile config.properties
inference_address=http://0.0.0.0:8081
management_address=http://0.0.0.0:8082
metrics_address=http://0.0.0.0:8083
model_store=model_store
disable_token_authorization=true

Overwriting config.properties


In [20]:
!pip -q install torchserve torch-model-archiver nvgpu

!mkdir -p model_store
!python save_model.py

!torch-model-archiver \
  --model-name simplemodel \
  --version 1.0 \
  --model-file model.py \
  --serialized-file model.pth \
  --handler handler.py \
  --export-path model_store \
  --force

!ls -lah model_store

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 7.0 MB/s eta 0:00:00
/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:65: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
Saved model weights to model.pth
WARNING - Overwriting model_store/simplemodel.mar ...
total 12K
drwxr-xr-x 2 root root 4.0K Mar  1 20:36 .
drwxr-xr-x 1 root root 4.0K Mar  1 20:37 ..
-rw-r--r-- 1 root root 2.4K Mar  1 20:39 simplemodel.mar


In [21]:
!torchserve --start --ncs --ts-config config.properties --model-store model_store --models simplemodel=simplemodel.mar

Removing orphan pid file.
2026-03-01T20:40:06,984 [DEBUG] main org.pytorch.serve.util.ConfigManager - xpu-smi not available or failed: Cannot run program "xpu-smi": error=2, No such file or directory
2026-03-01T20:40:06,988 [WARN ] main org.pytorch.serve.util.ConfigManager - Your torchserve instance can access any URL to load models. When deploying to production, make sure to limit the set of allowed_urls in config.properties
2026-03-01T20:40:06,999 [INFO ] main org.pytorch.serve.servingsdk.impl.PluginsManager - Initializing plugins manager...
2026-03-01T20:40:07,056 [INFO ] main org.pytorch.serve.metrics.configuration.MetricConfiguration - Successfully loaded metrics configuration from /usr/local/lib/python3.12/dist-packages/ts/configs/metrics.yaml
2026-03-01T20:40:07,145 [INFO ] main org.pytorch.serve.ModelServer - 
Torchserve version: 0.12.0
TS Home: /usr/local/lib/python3.12/dist-packages
Current directory: /content
Temp directory: /tmp
Metrics config path: /usr/local/lib/python3.1

In [ ]:
import requests

url = "http://127.0.0.1:8081/predictions/simplemodel"
payload = {"data": [0.0]*10}
r = requests.post(url, json=payload, timeout=10)
r.status_code, r.text # (200, '-0.09556816518306732')

Modify the script to include a `handler.py` file for TorchServe deployment and test the deployed API.

---

### Submission
Submit your completed notebook with answers and executed code outputs. Ensure that all model outputs, logs, and deployment results are included in your submission.
